In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

# 数据集

In [2]:
words = open("names.txt", 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# 老师的实现，更像个滚动窗口
block_size = 3  # context length: 预测下一个字符时所使用的上文长度
X,Y = [], []
for w in words[:5]:
    print(w)
    context = [0]*block_size  # context 用来存放当前上下文的变量
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ---> {ch}")
        context = context[1:] + [ix]  #上下文向右滑动一位 crop and append
X = torch.tensor(X)
Y= torch.tensor(Y)
print(X.shape, Y.shape, X.dtype, Y.dtype)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
torch.Size([32, 3]) torch.Size([32]) torch.int64 torch.int64


In [5]:
# 构建用于MLP的数据集 这里输入变成3个字符 输出下一个字符 即：用前三个字符预测下一个字符

# 我的实现，其实结果是一样的，但是给人的含义不同
block_size = 3  # context length: 预测下一个字符时所使用的上文长度
X,Y = [], []
for w in words[:5]:
    print(w)
    context = ['.', '.', '.'] + list(w) + ['.']  
    # 这里这个block_size=3 其实就是输入的窗口长度，加上一个输出，其实整体的窗口长度是4，为了防止出现长度小于4的样本，这里直接对所有样本全部填充了...（这里的 . 字符其实是空白字符，占位用的）
    for i in range(len(context)-block_size):
        in_chs = context[i:i+block_size]
        out_ch = context[i+block_size]
        in_idx = [stoi[ch] for ch in in_chs]
        out_idx = stoi[out_ch]
        X.append(in_idx)
        Y.append(out_idx)
        print(f"{''.join(in_chs)} ---> {out_ch}")
X = torch.tensor(X)
Y= torch.tensor(Y)
print(X.shape, Y.shape)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
torch.Size([32, 3]) torch.Size([32])


# 构建查找表

原始论文是 1.7w个单词，嵌入到30维的空间里，即： 每个单词是一个30维的向量表示

这里因为全部的词表只有27个字符，因此简单点，先使用2维表示

注意，在之前的Bigram/单个线性层中，可以理解为：一个字符是由一个27维的向量表示的 

In [7]:
C = torch.randn((27,2))

In [13]:
C[5]

tensor([-0.6772, -0.8491])

In [12]:
r = F.one_hot(torch.tensor([5]), num_classes=27).float()@C
r, r.shape

(tensor([[-0.6772, -0.8491]]), torch.Size([1, 2]))

In [11]:
F.one_hot(torch.tensor([1]), num_classes=27).shape

torch.Size([1, 27])

这里要说明的问题就是:
+ 对于从嵌入矩阵中取出某个index的嵌入向量来说
+ 直接用序号索引, 或者直接用one-hot效果是一样的
+ 因为one-hot就是依据序号索引生成
+ 这里更多想说明:  矩阵乘法的作用

In [34]:
import numpy as np
index_num = np.random.randint(0, 27, 1000).tolist()

In [35]:
%time
r = F.one_hot(torch.tensor(index_num), num_classes=27).float()@C

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 5.72 µs


In [36]:
%time
r = C[index_num,:]

CPU times: user 2 µs, sys: 1e+03 ns, total: 3 µs
Wall time: 6.2 µs


`索引操作`比 `one-hot + 矩阵乘法`快得多，通常会快 10倍 到 50倍 以上。 `one-hot + 矩阵乘法`是深度学习中典型的“反模式”（做了大量无用功）。

如果采用这种方式, 也只是为了把这个操作变成一个`Wx`的乘法操作, 可以作为一个没有激活函数的线性层, 放到神经网络中, 

可以对`Wx`(也就是上面的`C`)进行反向传播, 来优化这个嵌入矩阵

In [37]:
X[1]

tensor([0, 0, 5])

In [38]:
C[X[1]]

tensor([[ 0.0660,  0.0489],
        [ 0.0660,  0.0489],
        [-0.6772, -0.8491]])

In [40]:
C[X].shape, C.shape, X.shape

(torch.Size([32, 3, 2]), torch.Size([27, 2]), torch.Size([32, 3]))

这个索引就很牛逼了, 